# scikit-learn 나이브 베이즈 스팸 메일 분류 예제

메일 내용을 입력받아 **정상 메일(0)** 또는 **스팸 메일(1)**로 분류한다.

사용 알고리즘은 다음과 같다.

- `CountVectorizer`: 메일의 단어를 출현 횟수 벡터로 변환한다.
- `MultinomialNB`: 단어 빈도에 적합한 다항 나이브 베이즈 분류기이다.

나이브 베이즈는 클래스별 단어 출현 확률을 학습하고 다음 값이 더 큰 클래스를 선택한다.

$$P(C\mid x_1, x_2, \ldots, x_n) \propto P(C)\prod_{i=1}^{n}P(x_i\mid C)$$

## 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)

# 운영체제에 설치된 한글 글꼴을 자동 선택한다.
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42

## 2. 학습용 메일 데이터 만들기

외부 파일 없이 실행할 수 있도록 교육용 메일 40개를 사용한다. `label`이 1이면 스팸, 0이면 정상이다.

In [ ]:
spam_mails = [
    '무료 쿠폰에 당첨되었습니다 지금 링크를 클릭하세요',
    '축하합니다 현금 경품 대상자로 선정되었습니다',
    '신용등급 무관 저금리 대출 즉시 신청 가능',
    '오늘만 초특가 할인 광고 지금 구매하세요',
    '무료 여행권 수령을 위해 주소를 입력하세요',
    '미수령 포인트가 있습니다 링크에서 확인하세요',
    '고수익 투자 정보를 단독으로 공개합니다',
    '계정이 정지됩니다 비밀번호를 입력하세요',
    '상품권을 즉시 지급하는 이벤트에 참여하세요',
    '대출 한도 조회 시 사은품을 드립니다',
    '로또 당첨 번호와 상금을 확인하세요',
    '무료 체험 기회가 오늘 마감됩니다',
    '세금 환급금을 받으려면 계좌를 등록하세요',
    '비정상 접속이 발견되었습니다 보안 링크를 누르세요',
    '선착순 경품 이벤트 지금 응모하세요',
    '보험 무료 상담 후 현금 사은품 지급',
    '원금 보장 코인 투자로 매일 수익을 얻으세요',
    '택배 주소 오류입니다 링크에서 수정하세요',
    '카드 발급 시 현금 보너스를 즉시 드립니다',
    '당첨 상품 배송을 위해 개인정보를 회신하세요'
]

normal_mails = [
    '내일 프로젝트 회의 일정을 공유합니다',
    '주간 업무 보고서를 첨부했습니다',
    '계약서 수정본을 검토해 주세요',
    '다음 주 교육 일정과 강의실을 안내합니다',
    '지난달 세금계산서를 보내드립니다',
    '고객 문의에 대한 답변입니다',
    '연구 계획서 초안을 공유합니다',
    '토요일 서버 정기 점검이 진행됩니다',
    '면접 일정이 다음 주로 확정되었습니다',
    '오늘 회의 내용을 정리해 전달합니다',
    '주문한 사무용품이 내일 배송됩니다',
    '논문 심사 의견을 반영해 주세요',
    '신청하신 휴가가 승인되었습니다',
    '분기 예산안을 검토하고 의견을 주세요',
    '상담 예약 시간을 확인드립니다',
    '새 기능의 테스트를 요청드립니다',
    '워크숍 장소가 강의실로 변경되었습니다',
    '월간 실적 발표 자료를 공유합니다',
    '신규 직원 계정이 생성되었습니다',
    '데이터 분석 결과를 첨부합니다'
]

mail_df = pd.DataFrame({
    'text': spam_mails + normal_mails,
    'label': [1] * len(spam_mails) + [0] * len(normal_mails)
})
mail_df['class'] = mail_df['label'].map({0: '정상', 1: '스팸'})
mail_df = mail_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'전체 메일 수: {len(mail_df)}개')
mail_df.head()

## 3. 학습 데이터와 테스트 데이터 분리

학습 데이터는 모델이 단어별 확률을 계산하는 데 사용하고, 테스트 데이터는 학습하지 않은 메일에 대한 성능을 확인하는 데 사용한다. `stratify`로 정상·스팸 비율을 동일하게 유지한다.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    mail_df['text'],
    mail_df['label'],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=mail_df['label']
)

print(f'학습 데이터: {len(X_train_text)}개')
print(f'테스트 데이터: {len(X_test_text)}개')
print('테스트 클래스 분포:', y_test.value_counts().sort_index().to_dict())

## 4. 메일을 단어 빈도 벡터로 변환

`fit_transform()`은 학습 메일에서 단어 사전을 만들고 각 단어의 출현 횟수를 계산한다. 테스트 메일에는 학습된 사전을 그대로 적용해야 하므로 `transform()`만 사용한다.

In [ ]:
vectorizer = CountVectorizer(
    token_pattern=r'(?u)\b\w+\b',
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print('학습 행렬 크기:', X_train.shape)
print('테스트 행렬 크기:', X_test.shape)
print('단어 및 구문 특징 수:', len(vectorizer.get_feature_names_out()))

## 5. MultinomialNB 모델 학습

`alpha=1.0`은 Laplace smoothing을 의미한다. 학습 데이터에서 특정 단어가 한 클래스에 한 번도 나오지 않아 확률이 0이 되는 문제를 방지한다.

In [ ]:
model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)
print('모델 학습 완료')

## 6. 테스트 메일 분류 및 성능 평가

In [ ]:
y_pred = model.predict(X_test)
y_spam_probability = model.predict_proba(X_test)[:, 1]

print(f'정확도: {accuracy_score(y_test, y_pred):.3f}')
print('\n분류 리포트')
print(classification_report(
    y_test, y_pred,
    labels=[0, 1], target_names=['정상', '스팸'],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['정상', '스팸']).plot(
    cmap='Blues', values_format='d'
)
plt.title('스팸 메일 분류 혼동행렬')
plt.show()

In [ ]:
result_df = pd.DataFrame({
    '메일': X_test_text.values,
    '실제': ['스팸' if value == 1 else '정상' for value in y_test.values],
    '예측': ['스팸' if value == 1 else '정상' for value in y_pred],
    '스팸 확률': y_spam_probability.round(3)
})
result_df['정답'] = result_df['실제'] == result_df['예측']
result_df.sort_values('스팸 확률', ascending=False)

## 7. 새로운 메일이 스팸인지 예측

새로운 메일도 학습에 사용한 `vectorizer`로 변환한 뒤 예측한다.

In [ ]:
new_mails = [
    '무료 경품에 당첨되었습니다 링크를 클릭하세요',
    '내일 프로젝트 회의 자료를 보내드립니다',
    '저금리 대출 한도를 지금 확인하세요',
    '다음 주 교육 일정이 변경되었습니다'
]

new_vectors = vectorizer.transform(new_mails)
new_predictions = model.predict(new_vectors)
new_probabilities = model.predict_proba(new_vectors)[:, 1]

new_result = pd.DataFrame({
    '새 메일': new_mails,
    '분류 결과': ['스팸' if value == 1 else '정상' for value in new_predictions],
    '스팸 확률': new_probabilities.round(3)
})
new_result

## 전체 과정 정리

```text
메일 텍스트 준비
       ↓
CountVectorizer로 단어 빈도 변환
       ↓
MultinomialNB 학습
       ↓
정상·스팸 확률 비교
       ↓
확률이 높은 클래스로 분류
```

이 데이터는 알고리즘 이해를 위한 소규모 교육용 예시이다. 실제 적용 시에는 충분한 실제 메일, 개인정보 비식별화, 교차검증, 피싱 URL 및 발신자 정보 등의 추가 특징이 필요하다.